In [ ]:


# 1. Mount Drive FIRST
from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive mounted!")

# 2. Imports
import os, json
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
print("✅ Imports done!")

# 3. Download dataset
os.environ['KAGGLE_TOKEN'] = 'KGAT_819e6770236c198776b42d09083d8aed'  # ← paste token
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({"username": "osakiki",  # ← paste username
               "key": os.environ['KAGGLE_TOKEN']}, f)
os.system('chmod 600 /root/.kaggle/kaggle.json')
os.system('kaggle datasets download -d paultimothymooney/chest-xray-pneumonia')
os.system('unzip -q chest-xray-pneumonia.zip')
print("✅ Dataset ready!")

# 4. Data generators
train_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10, horizontal_flip=True,
    zoom_range=0.1, width_shift_range=0.1,
    height_shift_range=0.1, validation_split=0.1
)
val_gen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_data = train_gen.flow_from_directory(
    'chest_xray/train', target_size=(224,224),
    batch_size=32, class_mode='binary',
    color_mode='rgb', subset='training', seed=42
)
val_data = train_gen.flow_from_directory(
    'chest_xray/train', target_size=(224,224),
    batch_size=32, class_mode='binary',
    color_mode='rgb', subset='validation', seed=42
)
test_data = val_gen.flow_from_directory(
    'chest_xray/test', target_size=(224,224),
    batch_size=32, class_mode='binary',
    color_mode='rgb', shuffle=False
)
print("✅ Data generators ready!")

# 5. Build VGG16
def build_model(input_shape=(224,224,3)):
    base = VGG16(weights='imagenet', include_top=False, input_shape=input_shape)
    base.trainable = False
    inputs = layers.Input(input_shape)
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    return Model(inputs, outputs, name='VGG16_Pneumonia')

model_v2 = build_model()
model_v2.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.AUC(name='auc'),
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)
print("✅ VGG16 built!")

# 6. Class weights
n_normal   = len(os.listdir('chest_xray/train/NORMAL'))
n_pneumonia = len(os.listdir('chest_xray/train/PNEUMONIA'))
total = n_normal + n_pneumonia
class_weights = {0: total/(2*n_normal), 1: total/(2*n_pneumonia)}
print(f"Class weights: {class_weights}")

# 7. Create Drive folder
save_path = '/content/drive/MyDrive/chest_xray_project/model_v2/best_vgg16.h5'
os.makedirs(os.path.dirname(save_path), exist_ok=True)
print(f"✅ Save folder ready: {os.path.dirname(save_path)}")

# 8. Callbacks — saves DIRECTLY to Drive!!
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_auc', patience=5,
        mode='max', restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        save_path, monitor='val_auc',
        mode='max', save_best_only=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, verbose=1
    )
]

# 9. TRAIN!!
print("\n🚀 Training VGG16...")
history = model_v2.fit(
    train_data, validation_data=val_data,
    epochs=20, callbacks=callbacks,
    class_weight=class_weights, verbose=1
)
print("✅ Training done!!")

# 10. Verify save!!
exists = os.path.exists(save_path)
size   = os.path.getsize(save_path) / 1024 / 1024 if exists else 0
print(f"\n💾 Saved to Drive: {exists}")
print(f"📁 File size: {size:.1f} MB")
print("\n🎉 DONE!! Never retrain again!!")

Mounted at /content/drive
✅ Drive mounted!
✅ Imports done!
✅ Dataset ready!
Found 4695 images belonging to 2 classes.
Found 521 images belonging to 2 classes.
Found 624 images belonging to 2 classes.
✅ Data generators ready!
58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
✅ VGG16 built!
Class weights: {0: 1.9448173005219984, 1: 0.6730322580645162}
✅ Save folder ready: /content/drive/MyDrive/chest_xray_project/model_v2

🚀 Training VGG16...
Epoch 1/20
147/147 ━━━━━━━━━━━━━━━━━━━━ 0s 777ms/step - accuracy: 0.6380 - auc: 0.6796 - loss: 1.2971 - precision: 0.8289 - recall: 0.6448
Epoch 1: val_auc improved from None to 0.96442, saving model to /content/drive/MyDrive/chest_xray_project/model_v2/best_vgg16.h5



Epoch 1: finished saving model to /content/drive/MyDrive/chest_xray_project/model_v2/best_vgg16.h5
147/147 ━━━━━━━━━━━━━━━━━━━━ 153s 921ms/step - accuracy: 0.7150 - auc: 0.7913 - loss: 0.8676 - precision: 0.8788 - recall: 0.7150 - val_accuracy: 0.8887 - val_auc: 0.9644 - val_loss: 0.3100 - val_precision: 0.9881 - val_recall: 0.8605 - learning_rate: 1.0000e-04
Epoch 2/20
147/147 ━━━━━━━━━━━━━━━━━━━━ 0s 691ms/step - accuracy: 0.8318 - auc: 0.8967 - loss: 0.5192 - precision: 0.9240 - recall: 0.8420
Epoch 2: val_auc improved from 0.96442 to 0.96520, saving model to /content/drive/MyDrive/chest_xray_project/model_v2/best_vgg16.h5



Epoch 2: finished saving model to /content/drive/MyDrive/chest_xray_project/model_v2/best_vgg16.h5
147/147 ━━━━━━━━━━━━━━━━━━━━ 114s 773ms/step - accuracy: 0.8413 - auc: 0.9112 - loss: 0.4578 - precision: 0.9353 - recall: 0.8449 - val_accuracy: 0.8772 - val_auc: 0.9652 - val_loss: 0.3522 - val_precision: 0.9939 - val_recall: 0.8398 - learning_rate: 1.0000e-04
Epoch 3/20
147/147 ━━━━━━━━━━━━━━━━━━━━ 0s 697ms/step - accuracy: 0.8664 - auc: 0.9361 - loss: 0.3576 - precision: 0.9469 - recall: 0.8701
Epoch 3: val_auc improved from 0.96520 to 0.97875, saving model to /content/drive/MyDrive/chest_xray_project/model_v2/best_vgg16.h5



Epoch 3: finished saving model to /content/drive/MyDrive/chest_xray_project/model_v2/best_vgg16.h5
147/147 ━━━━━━━━━━━━━━━━━━━━ 125s 848ms/step - accuracy: 0.8750 - auc: 0.9424 - loss: 0.3343 - precision: 0.9506 - recall: 0.8773 - val_accuracy: 0.9098 - val_auc: 0.9787 - val_loss: 0.2274 - val_precision: 0.9857 - val_recall: 0.8915 - learning_rate: 1.0000e-04
Epoch 4/20
147/147 ━━━━━━━━━━━━━━━━━━━━ 0s 693ms/step - accuracy: 0.8841 - auc: 0.9542 - loss: 0.2838 - precision: 0.9618 - recall: 0.8797
Epoch 4: val_auc did not improve from 0.97875
147/147 ━━━━━━━━━━━━━━━━━━━━ 114s 771ms/step - accuracy: 0.8924 - auc: 0.9574 - loss: 0.2714 - precision: 0.9657 - recall: 0.8868 - val_accuracy: 0.9098 - val_auc: 0.9773 - val_loss: 0.2154 - val_precision: 0.9830 - val_recall: 0.8941 - learning_rate: 1.0000e-04
Epoch 5/20
147/147 ━━━━━━━━━━━━━━━━━━━━ 0s 687ms/step - accuracy: 0.9102 - auc: 0.9633 - loss: 0.2526 - precision: 0.9644 - recall: 0.9121
Epoch 5: val_auc improved from 0.97875 to 0.98333,


Epoch 5: finished saving model to /content/drive/MyDrive/chest_xray_project/model_v2/best_vgg16.h5
147/147 ━━━━━━━━━━━━━━━━━━━━ 114s 778ms/step - accuracy: 0.9110 - auc: 0.9645 - loss: 0.2470 - precision: 0.9677 - recall: 0.9106 - val_accuracy: 0.9021 - val_auc: 0.9833 - val_loss: 0.2367 - val_precision: 0.9912 - val_recall: 0.8760 - learning_rate: 1.0000e-04
Epoch 6/20
147/147 ━━━━━━━━━━━━━━━━━━━━ 0s 690ms/step - accuracy: 0.9170 - auc: 0.9721 - loss: 0.2170 - precision: 0.9700 - recall: 0.9155
Epoch 6: val_auc improved from 0.98333 to 0.98785, saving model to /content/drive/MyDrive/chest_xray_project/model_v2/best_vgg16.h5



Epoch 6: finished saving model to /content/drive/MyDrive/chest_xray_project/model_v2/best_vgg16.h5
147/147 ━━━━━━━━━━━━━━━━━━━━ 115s 780ms/step - accuracy: 0.9152 - auc: 0.9702 - loss: 0.2238 - precision: 0.9716 - recall: 0.9126 - val_accuracy: 0.9175 - val_auc: 0.9879 - val_loss: 0.2279 - val_precision: 1.0000 - val_recall: 0.8889 - learning_rate: 1.0000e-04
Epoch 7/20
147/147 ━━━━━━━━━━━━━━━━━━━━ 0s 690ms/step - accuracy: 0.9150 - auc: 0.9699 - loss: 0.2238 - precision: 0.9700 - recall: 0.9133
Epoch 7: val_auc did not improve from 0.98785
147/147 ━━━━━━━━━━━━━━━━━━━━ 113s 769ms/step - accuracy: 0.9208 - auc: 0.9731 - loss: 0.2103 - precision: 0.9730 - recall: 0.9189 - val_accuracy: 0.9117 - val_auc: 0.9859 - val_loss: 0.2068 - val_precision: 0.9942 - val_recall: 0.8863 - learning_rate: 1.0000e-04
Epoch 8/20
147/147 ━━━━━━━━━━━━━━━━━━━━ 0s 689ms/step - accuracy: 0.9219 - auc: 0.9716 - loss: 0.2212 - precision: 0.9741 - recall: 0.9187
Epoch 8: val_auc did not improve from 0.98785
147/

In [ ]:
!pip install flask flask-cors -q
print("✅ Flask and CORS installed!")

from flask import Flask, request, jsonify
from flask_cors import CORS
import cv2, base64, threading
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.vgg16 import preprocess_input

app = Flask(__name__)
CORS(app)

def img_to_base64(img_array):
    _, buffer = cv2.imencode('.jpg', img_array)
    return base64.b64encode(buffer).decode('utf-8')

def base64_to_img(base64_str):
    img_data = base64.b64decode(base64_str.split(',')[1])
    np_arr = np.frombuffer(img_data, np.uint8)
    return cv2.imdecode(np_arr, cv2.IMREAD_COLOR)

def get_gradcam(inp, model):
    vgg16 = model.get_layer('vgg16')
    grad_model = tf.keras.models.Model(
        inputs=vgg16.input,
        outputs=[vgg16.get_layer('block5_conv3').output, vgg16.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, vgg_output = grad_model(inp)
        x = model.get_layer('global_average_pooling2d')(vgg_output)
        x = model.get_layer('dense')(x)
        x = model.get_layer('dropout')(x, training=False)
        x = model.get_layer('dense_1')(x)
        x = model.get_layer('dropout_1')(x, training=False)
        predictions = model.get_layer('dense_2')(x)
        loss = predictions[:, 0]
    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = conv_outputs[0] @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = np.maximum(heatmap.numpy(), 0)
    heatmap = heatmap / (heatmap.max() + 1e-8)
    return heatmap

@app.route('/analyze', methods=['POST'])
def analyze():
    try:
        data = request.get_json()
        img_bgr = base64_to_img(data['image'])
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_resized = cv2.resize(img_rgb, (224, 224))
        inp = preprocess_input(img_resized.copy().astype(np.float32))
        inp = np.expand_dims(inp, axis=0)

        pred = float(model_v2.predict(inp, verbose=0)[0][0])
        is_pneumonia = pred > 0.5
        confidence = pred if is_pneumonia else 1 - pred

        heatmap = get_gradcam(inp, model_v2)
        heatmap_resized = cv2.resize(heatmap, (224, 224))
        heatmap_colored = cv2.applyColorMap(
            (heatmap_resized * 255).astype(np.uint8), cv2.COLORMAP_JET
        )
        overlay = cv2.addWeighted(
            cv2.cvtColor(img_resized, cv2.COLOR_RGB2BGR),
            0.6, heatmap_colored, 0.4, 0
        )

        notes = (
            "<strong>Findings:</strong> The model identified opacification patterns "
            "consistent with pneumonia, primarily in the lower lung zones. Grad-CAM "
            "highlights (red regions) correspond to areas of increased radiodensity "
            "in the original X-ray.<br><br>"
            "<strong>Recommendation:</strong> Correlate with clinical symptoms and "
            "laboratory findings. Further imaging may be warranted."
        ) if is_pneumonia else (
            "<strong>Findings:</strong> Lung fields appear clear with no significant "
            "consolidation or opacification. Cardiac silhouette within normal limits. "
            "Grad-CAM activation is diffuse, confirming absence of focal infection "
            "patterns.<br><br>"
            "<strong>Recommendation:</strong> Continue routine follow-up as clinically "
            "indicated."
        )

        return jsonify({
            'label': 'PNEUMONIA' if is_pneumonia else 'NORMAL',
            'confidence': round(confidence * 100),
            'is_pneumonia': is_pneumonia,
            'notes': notes,
            'orig_img': f'data:image/jpeg;base64,{img_to_base64(cv2.cvtColor(img_resized, cv2.COLOR_RGB2BGR))}',
            'heatmap_img': f'data:image/jpeg;base64,{img_to_base64(heatmap_colored)}',
            'overlay_img': f'data:image/jpeg;base64,{img_to_base64(overlay)}',
        })
    except Exception as e:
        return jsonify({'error': str(e)}), 500

@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok'})

def run_flask():
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

thread = threading.Thread(target=run_flask)
thread.daemon = True
thread.start()
print("✅ Flask running!")

✅ Flask and CORS installed!
✅ Flask running!


In [ ]:
!pip install pyngrok -q
print("✅ pyngrok installed!")

from pyngrok import ngrok

ngrok.set_auth_token("3FvGrYipdxgGfeKjXKYx0KCxfLT_BWmFFwrQc5Nstfk8jwhv")
public_url = ngrok.connect(5000)
print(f"✅ Your URL: {public_url}")
print(f"🔗 Update HTML with: {public_url}")

✅ pyngrok installed!
✅ Your URL: NgrokTunnel: "https://enquirer-ambitious-gilled.ngrok-free.dev" -> "http://localhost:5000"
🔗 Update HTML with: NgrokTunnel: "https://enquirer-ambitious-gilled.ngrok-free.dev" -> "http://localhost:5000"


In [ ]:
from google.colab import files
files.download('chest_xray/test/PNEUMONIA/person1_virus_6.jpeg')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
import shutil

shutil.copy(
    '/content/drive/MyDrive/chest_xray_project/model_v2/best_vgg16.h5',
    'best_vgg16.h5'
)
files.download('best_vgg16.h5')
print("✅ Model downloading!!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Model downloading!!
